In [1]:

import os
import warnings
 
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
 
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import ndarrays_to_parameters, Context, NDArrays, Scalar
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
 
from typing import List, Tuple
 
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)
 
# =============================================================================
# 0. KONFIGURASI GLOBAL (satu sumber kebenaran — jangan didefinisikan ulang
#    di bagian bawah notebook, cukup ubah di sini)
# =============================================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DATA_PATH = "../dataset/drebin_dataset.csv"   # <-- sesuaikan path dataset Anda
LABEL_COLUMN = "class"
 
NUM_CLIENTS = 4           # jumlah client federated
NUM_ROUNDS = 10            # jumlah round komunikasi federated
LOCAL_EPOCHS = 3           # epoch lokal per client per round (diperbaiki, sebelumnya 50 -> overfit/lambat)
BATCH_SIZE = 32
LEARNING_RATE = 0.01
PROXIMAL_MU = 0.1          # koefisien mu untuk FedProx
DIRICHLET_ALPHA = 0.3      # semakin kecil -> semakin non-IID (skewed)
TEST_SIZE = 0.15           # porsi data uji global (held-out, dievaluasi di server)
 
PARTITION_MODE = "non_iid"  # "iid" | "non_iid"
 
# Device dipaksa CPU
DEVICE = torch.device("cpu")
NUM_GPUS_AVAILABLE = 0  # dipaksa 0 supaya alokasi resource simulasi juga tidak memakai GPU
 
print(f"[INFO] Device yang digunakan: {DEVICE}")
print("[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).")


2026-09-22 15:20:23,649	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


[INFO] Device yang digunakan: cpu
[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).


In [2]:

# %% [markdown]
# ## 1. Load & Bersihkan Dataset
 
# %%
data_raw = pd.read_csv(DATA_PATH)
data = data_raw.copy()
 
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])
 
SPECIAL_COLUMN = "TelephonyManager.getSimCountryIso"
if SPECIAL_COLUMN in data.columns:
    data[SPECIAL_COLUMN] = data[SPECIAL_COLUMN].replace("?", np.nan)
    data[SPECIAL_COLUMN] = pd.to_numeric(data[SPECIAL_COLUMN], errors="coerce")
 
if LABEL_COLUMN not in data.columns:
    raise KeyError(f"Kolom '{LABEL_COLUMN}' tidak ditemukan. Kolom tersedia: {list(data.columns)}")
 
data[LABEL_COLUMN] = data[LABEL_COLUMN].replace({"B": 0, "S": 1})
 
feature_columns = [c for c in data.columns if c != LABEL_COLUMN]
data[feature_columns] = data[feature_columns].apply(pd.to_numeric, errors="coerce")
 
data = data.dropna().reset_index(drop=True)
 
X = data.drop(columns=[LABEL_COLUMN]).values.astype(np.float32)
y = data[LABEL_COLUMN].astype(int).values
 
N_CLASSES = len(np.unique(y))  # <-- FIX: sebelumnya tidak pernah didefinisikan, menyebabkan NameError
 
print("Ukuran X:", X.shape, "| Ukuran y:", y.shape)
print("Jumlah kelas:", N_CLASSES)
print("Distribusi kelas:", np.bincount(y))
 
# %% [markdown]
# ## 2. Split Data: Train (full, untuk FL), Validation, Test
 
# %%
# Split pertama: pisahkan test set global (held-out, dievaluasi di server)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
# X_train_full / y_train_full inilah yang FIX-nya dipakai untuk partisi client di bawah
# (sebelumnya notebook memakai nama ini tanpa pernah mendefinisikannya)
 
# Split kedua: dari sisa data train, ambil validation set (opsional, mis. untuk tuning terpusat)
val_ratio = TEST_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=val_ratio,
    random_state=SEED,
    stratify=y_train_full,
)
 
# Scaler di-fit pada data train saja, lalu diterapkan ke semua split
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
X_train_full_scaled = scaler.transform(X_train_full)  # dipakai untuk partisi FL
 
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
 
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)
 
print("Train:", X_train_t.shape, "| Val:", X_val_t.shape, "| Test:", X_test_t.shape)
print("Train full (untuk partisi FL):", X_train_full_scaled.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/drebin_dataset.csv'

In [ ]:

# %% [markdown]
# ## 3. Fungsi Partisi Data: IID dan Non-IID
 
# %%
def partition_iid(
    X: np.ndarray, y: np.ndarray, num_clients: int, seed: int = SEED
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Bagi data secara acak merata (IID) ke semua client."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    rng.shuffle(idx)
    splits = np.array_split(idx, num_clients)
    return [(X[s], y[s]) for s in splits]
 
 
def partition_non_iid_dirichlet(
    X: np.ndarray,
    y: np.ndarray,
    num_clients: int,
    alpha: float = 0.3,
    seed: int = SEED,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Bagi data secara non-IID menggunakan distribusi Dirichlet per kelas.
    alpha kecil (misal 0.1-0.5) -> distribusi label makin timpang antar client.
    alpha besar (misal 10+) -> mendekati IID.
    """
    rng = np.random.default_rng(seed)
    n_classes = len(np.unique(y))
    client_indices = [[] for _ in range(num_clients)]
 
    for c in range(n_classes):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
 
        proportions = rng.dirichlet(alpha=np.repeat(alpha, num_clients))
        cut_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, cut_points)
 
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())
 
    partitions = []
    for indices in client_indices:
        indices = np.array(indices, dtype=int)
        rng.shuffle(indices)
        partitions.append((X[indices], y[indices]))
 
    return partitions
 
 
def print_partition_summary(partitions: List[Tuple[np.ndarray, np.ndarray]], title: str):
    print(f"\n================ {title} ================")
    for i, (Xi, yi) in enumerate(partitions):
        counts = np.bincount(yi, minlength=N_CLASSES)  # N_CLASSES kini sudah didefinisikan di atas
        print(f"Client {i}: total={len(yi)} | kelas 0={counts[0]} | kelas 1={counts[1]}")
 
 
def show_partition_stats(partitions, name="Partition"):
    print("=" * 70)
    print(f" {name}")
    print("=" * 70)
 
    total_data = sum(len(y_client) for _, y_client in partitions)
 
    for client_id, (X_client, y_client) in enumerate(partitions):
        unique, counts = np.unique(y_client, return_counts=True)
        label_counts = dict(zip(unique, counts))
 
        count_0 = label_counts.get(0, 0)
        count_1 = label_counts.get(1, 0)
 
        pct_0 = count_0 / len(y_client) * 100
        pct_1 = count_1 / len(y_client) * 100
 
        print(
            f"Client {client_id + 1}: "
            f"Total={len(y_client):5d} | "
            f"Label 0={count_0:5d} ({pct_0:6.2f}%) | "
            f"Label 1={count_1:5d} ({pct_1:6.2f}%)"
        )
 
    print("-" * 70)
    print(f"Total data : {total_data}")
    print("=" * 70)
 
 
# %% [markdown]
# ## 4. Membuat Partisi Sesuai PARTITION_MODE
# (Menggunakan X_train_full_scaled/y_train_full yang sudah didefinisikan di Bagian 2,
#  bukan variabel yang tidak pernah dibuat seperti pada skrip asli.)
 
# %%
# Contoh melihat kedua mode partisi sekaligus (opsional, untuk perbandingan)
iid_partitions = partition_iid(
    X_train_full_scaled, y_train_full, num_clients=NUM_CLIENTS, seed=SEED
)
non_iid_partitions = partition_non_iid_dirichlet(
    X_train_full_scaled, y_train_full, num_clients=NUM_CLIENTS, alpha=DIRICHLET_ALPHA, seed=SEED
)
 
show_partition_stats(iid_partitions, name="IID Partition")
show_partition_stats(non_iid_partitions, name=f"Non-IID Dirichlet (alpha={DIRICHLET_ALPHA})")
 
# Partisi final yang benar-benar dipakai untuk simulasi FL, sesuai PARTITION_MODE di config atas
if PARTITION_MODE == "iid":
    client_partitions = iid_partitions
    print_partition_summary(client_partitions, "PARTISI IID (dipakai untuk FL)")
else:
    client_partitions = non_iid_partitions
    print_partition_summary(client_partitions, "PARTISI NON-IID Dirichlet (dipakai untuk FL)")

 IID Partition
Client 1: Total= 3194 | Label 0= 2035 ( 63.71%) | Label 1= 1159 ( 36.29%)
Client 2: Total= 3194 | Label 0= 2003 ( 62.71%) | Label 1= 1191 ( 37.29%)
Client 3: Total= 3194 | Label 0= 2008 ( 62.87%) | Label 1= 1186 ( 37.13%)
Client 4: Total= 3194 | Label 0= 2008 ( 62.87%) | Label 1= 1186 ( 37.13%)
----------------------------------------------------------------------
Total data : 12776
 Non-IID Dirichlet (alpha=0.3)
Client 1: Total=  334 | Label 0=  334 (100.00%) | Label 1=    0 (  0.00%)
Client 2: Total= 1472 | Label 0= 1398 ( 94.97%) | Label 1=   74 (  5.03%)
Client 3: Total= 2495 | Label 0= 1424 ( 57.07%) | Label 1= 1071 ( 42.93%)
Client 4: Total= 8475 | Label 0= 4898 ( 57.79%) | Label 1= 3577 ( 42.21%)
----------------------------------------------------------------------
Total data : 12776

================ PARTISI NON-IID Dirichlet (dipakai untuk FL) ================
Client 0: total=334 | kelas 0=334 | kelas 1=0
Client 1: total=1472 | kelas 0=1398 | kelas 1=74
Client 

In [ ]:

# %% [markdown]
# ## 5. Definisi Model (θ dan φ) — pakai nama layer eksplisit (fc1, fc2)
# supaya gampang diakses untuk analisis bobot nanti
 
# %%
class FeatureExtractor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.relu2 = nn.ReLU()
 
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return x
 
 
class ClassifierHead(nn.Module):
    def __init__(self, in_dim=32, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
 
    def forward(self, features):
        return self.fc(features)
 
 
class DrebinModel(nn.Module):
    def __init__(self, in_dim, num_classes=2):
        super().__init__()
        self.theta = FeatureExtractor(in_dim=in_dim)
        self.phi = ClassifierHead(num_classes=num_classes)
 
    def forward(self, x):
        features = self.theta(x)
        logits = self.phi(features)
        return logits
 
 
def get_parameters(model: nn.Module) -> NDArrays:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]
 
 
def set_parameters(model: nn.Module, parameters: NDArrays) -> None:
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)
 
 
def train(model: nn.Module, X: np.ndarray, y: np.ndarray, epochs: int, batch_size: int, device: torch.device = DEVICE):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
 
 
def test(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 64, device: torch.device = DEVICE) -> Tuple[float, Dict[str, float]]:
    """
    Evaluasi model dan hitung loss + accuracy + precision + recall + f1-score.
    Precision/recall/f1 dihitung dengan average="binary" (pos_label=1 -> kelas malware/S).
    Jika hanya ada 1 kelas pada batch data (edge case), fallback ke average="macro" dengan zero_division=0.
    """
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
 
    total_loss, correct, total = 0.0, 0, 0
    all_preds: List[int] = []
    all_labels: List[int] = []
 
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * len(yb)
            preds = torch.argmax(out, dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(yb.cpu().numpy().tolist())
 
    avg_loss = total_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
 
    n_unique = len(set(all_labels))
    avg_mode = "binary" if n_unique > 1 else "macro"
 
    precision = precision_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
    recall = recall_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
    f1 = f1_score(
        all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0
    )
 
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }
    return avg_loss, metrics
 
 
# %% [markdown]
# ### Membuat & Melihat Struktur Model
# FIX: sebelumnya ada `model = DrebinMLP(n_features=N_FEATURES, ...)` yang menimpa
# model DrebinModel yang benar. DrebinMLP dan N_FEATURES tidak pernah didefinisikan
# di skrip asli -> NameError. Di sini cukup satu kali pembuatan model, pakai
# in_dim = X_train_t.shape[1] dan N_CLASSES yang sudah dihitung di Bagian 1.
 
# %%
torch.manual_seed(SEED)
in_dim = X_train_t.shape[1]
model = DrebinModel(in_dim=in_dim, num_classes=N_CLASSES)
print(model)
 
# %%
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
 
print(f"Total parameter     : {total_params:,}")
print(f"Trainable parameter : {trainable_params:,}")
 
# %% [markdown]
# ### Melihat Parameter Model
 
# %%
parameters = get_parameters(model)
 
print("Jumlah parameter tensors:", len(parameters))
 
for i, param in enumerate(parameters):
    print(
        f"Parameter {i}: "
        f"shape={param.shape}, "
        f"dtype={param.dtype}"
    )
 
# %%
for i, param in enumerate(parameters):
    print(f"Parameter {i}:")
    print(param.flatten()[:10])
 
# %%
print("Isi state_dict model:")
 
for name, param in model.state_dict().items():
    print(
        f"{name:20s} "
        f"shape={tuple(param.shape)} "
        f"dtype={param.dtype}"
    )

DrebinModel(
  (theta): FeatureExtractor(
    (fc1): Linear(in_features=215, out_features=64, bias=True)
    (relu1): ReLU()
    (fc2): Linear(in_features=64, out_features=32, bias=True)
    (relu2): ReLU()
  )
  (phi): ClassifierHead(
    (fc): Linear(in_features=32, out_features=2, bias=True)
  )
)
Total parameter     : 15,970
Trainable parameter : 15,970
Jumlah parameter tensors: 6
Parameter 0: shape=(64, 215), dtype=float32
Parameter 1: shape=(64,), dtype=float32
Parameter 2: shape=(32, 64), dtype=float32
Parameter 3: shape=(32,), dtype=float32
Parameter 4: shape=(2, 32), dtype=float32
Parameter 5: shape=(2,), dtype=float32
Parameter 0:
[ 0.05214109  0.05660607 -0.01597725  0.06264877 -0.01494274  0.01376201
 -0.03320324  0.04005235  0.06012072 -0.05003303]
Parameter 1:
[-0.01203729  0.00048026  0.02825528 -0.03662788 -0.0202265  -0.05060738
 -0.04573953  0.03280409 -0.00570992 -0.03404211]
Parameter 2:
[-0.03886208 -0.04673357 -0.09200618 -0.12388533  0.05449936 -0.08064108
 -0.06

In [ ]:
# ### Membuat & Melihat Struktur Model
# FIX: sebelumnya ada `model = DrebinMLP(n_features=N_FEATURES, ...)` yang menimpa
# model DrebinModel yang benar. DrebinMLP dan N_FEATURES tidak pernah didefinisikan
# di skrip asli -> NameError. Di sini cukup satu kali pembuatan model, pakai
# in_dim = X_train_t.shape[1] dan N_CLASSES yang sudah dihitung di Bagian 1.
 
# %%
torch.manual_seed(SEED)
in_dim = X_train_t.shape[1]
model = DrebinModel(in_dim=in_dim, num_classes=N_CLASSES)
print(model)
 
# %%
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
 
print(f"Total parameter     : {total_params:,}")
print(f"Trainable parameter : {trainable_params:,}")
 
# %% [markdown]
# ### Melihat Parameter Model
 
# %%
parameters = get_parameters(model)
 
print("Jumlah parameter tensors:", len(parameters))
 
for i, param in enumerate(parameters):
    print(
        f"Parameter {i}: "
        f"shape={param.shape}, "
        f"dtype={param.dtype}"
    )
 
# %%
for i, param in enumerate(parameters):
    print(f"Parameter {i}:")
    print(param.flatten()[:10])
 
# %%
print("Isi state_dict model:")
 
for name, param in model.state_dict().items():
    print(
        f"{name:20s} "
        f"shape={tuple(param.shape)} "
        f"dtype={param.dtype}"
    )
 
 
# %% [markdown]
# ## 6. Testing Model dan Melihat Output
 
# %%
print("=" * 70)
print("MODEL INFORMATION")
print("=" * 70)
 
# FIX: DrebinMLP/N_FEATURES tidak pernah didefinisikan -> pakai DrebinModel/in_dim yang benar
model = DrebinModel(in_dim=in_dim, num_classes=N_CLASSES)
 
print(model)
 
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
 
 
# %% [markdown]
# ### Parameter Model
 
# %%
print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)
 
parameters = get_parameters(model)
 
for i, param in enumerate(parameters):
    print(
        f"Parameter {i}: "
        f"shape={param.shape}, "
        f"min={param.min():.6f}, "
        f"max={param.max():.6f}, "
        f"mean={param.mean():.6f}"
    )
 
 
# %% [markdown]
# ### Forward Pass
 
# %%
print("\n" + "=" * 70)
print("FORWARD PASS")
print("=" * 70)
 
# FIX: pakai X_train_full_scaled (sudah di-scale) supaya konsisten dengan X_test/X_val
X_sample = X_train_full_scaled[:5]
y_sample = y_train_full[:5]
 
X_tensor = torch.tensor(
    X_sample,
    dtype=torch.float32
)
 
model.eval()
 
with torch.no_grad():
    logits = model(X_tensor)
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=1)
 
print("Input shape       :", X_tensor.shape)
print("Logits shape      :", logits.shape)
 
print("\nLogits:")
print(logits)
 
print("\nProbabilities:")
print(probabilities)
 
print("\nPredictions :", predictions.numpy())
print("True labels :", y_sample)
 
 
# %%
print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)
 
train(
    model,
    X_train_full_scaled,   # FIX: versi scaled, konsisten dengan X_test
    y_train_full,
    epochs=5,
    batch_size=64
)
 
print("Training selesai!")
 
 
# %% [markdown]
# ### Testing
 
# %%
print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)
 
test_loss, test_metrics = test(
    model,
    X_test,
    y_test
)
 
print(f"Test Loss : {test_loss:.4f}")

DrebinModel(
  (theta): FeatureExtractor(
    (fc1): Linear(in_features=215, out_features=64, bias=True)
    (relu1): ReLU()
    (fc2): Linear(in_features=64, out_features=32, bias=True)
    (relu2): ReLU()
  )
  (phi): ClassifierHead(
    (fc): Linear(in_features=32, out_features=2, bias=True)
  )
)
Total parameter     : 15,970
Trainable parameter : 15,970
Jumlah parameter tensors: 6
Parameter 0: shape=(64, 215), dtype=float32
Parameter 1: shape=(64,), dtype=float32
Parameter 2: shape=(32, 64), dtype=float32
Parameter 3: shape=(32,), dtype=float32
Parameter 4: shape=(2, 32), dtype=float32
Parameter 5: shape=(2,), dtype=float32
Parameter 0:
[ 0.05214109  0.05660607 -0.01597725  0.06264877 -0.01494274  0.01376201
 -0.03320324  0.04005235  0.06012072 -0.05003303]
Parameter 1:
[-0.01203729  0.00048026  0.02825528 -0.03662788 -0.0202265  -0.05060738
 -0.04573953  0.03280409 -0.00570992 -0.03404211]
Parameter 2:
[-0.03886208 -0.04673357 -0.09200618 -0.12388533  0.05449936 -0.08064108
 -0.06

In [ ]:
 
def get_parameters(model: nn.Module) -> NDArrays:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]
 
 
def set_parameters(model: nn.Module, parameters: NDArrays) -> None:
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)
 
 
def train(model: nn.Module, X: np.ndarray, y: np.ndarray, epochs: int, batch_size: int, device: torch.device = DEVICE):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
 
 
def test(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 64, device: torch.device = DEVICE) -> Tuple[float, Dict[str, float]]:
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
 
    total_loss, correct, total = 0.0, 0, 0
    all_preds: List[int] = []
    all_labels: List[int] = []
 
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * len(yb)
            preds = torch.argmax(out, dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(yb.cpu().numpy().tolist())
 
    avg_loss = total_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
 
    n_unique = len(set(all_labels))
    avg_mode = "binary" if n_unique > 1 else "macro"
 
    precision = precision_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
    recall = recall_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
 
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }
    return avg_loss, metrics
 
 
# %% [markdown]
# ## 6. Flower Client (generik, menerima partisi lewat closure)
 
# %%
class DrebinClient(NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val, device: torch.device = DEVICE):
        self.model = DrebinMLP(N_FEATURES, N_CLASSES)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.device = device
 
    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        return get_parameters(self.model)
 
    def fit(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        train(
            self.model, self.X_train, self.y_train,
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, device=self.device,
        )
        return get_parameters(self.model), len(self.X_train), {}
 
    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        loss, metrics = test(self.model, self.X_val, self.y_val, device=self.device)
        return float(loss), len(self.X_val), metrics
 
 
def make_client_fn(client_partitions):
    """Membuat client_fn yang terikat ke satu set partisi tertentu (IID atau Non-IID)."""
    def client_fn(context: Context) -> Client:
        partition_id = int(context.node_config["partition-id"])
        X_client, y_client = client_partitions[partition_id]
 
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_client, y_client, test_size=0.2, random_state=SEED,
            stratify=y_client if len(np.unique(y_client)) > 1 else None,
        )
        return DrebinClient(X_tr, y_tr, X_val, y_val, device=DEVICE).to_client()
 
    return client_fn
 
 
def weighted_average(metrics: List[Tuple[int, Dict[str, Scalar]]]) -> Dict[str, Scalar]:
    """Agregasi weighted-average untuk accuracy, precision, recall, f1 dari semua client."""
    total_examples = sum(num_examples for num_examples, _ in metrics)
    aggregated: Dict[str, Scalar] = {}
    for key in ["accuracy", "precision", "recall", "f1_score"]:
        weighted_sum = sum(num_examples * m[key] for num_examples, m in metrics if key in m)
        aggregated[key] = weighted_sum / total_examples if total_examples > 0 else 0.0
    return aggregated
 
 
def make_evaluate_fn(X_test: np.ndarray, y_test: np.ndarray, history: List[Dict[str, float]], label: str):
    """
    Membuat evaluate_fn untuk server yang, selain mengevaluasi model global,
    juga MENYIMPAN metrik tiap round ke dalam `history` (dipakai untuk
    perbandingan IID vs Non-IID di akhir).
    """
    def evaluate(server_round: int, parameters: NDArrays, config: Dict[str, Scalar]):
        model = DrebinMLP(N_FEATURES, N_CLASSES)
        set_parameters(model, parameters)
        loss, metrics = test(model, X_test, y_test, device=DEVICE)
        print(
            f"[{label} | Round {server_round}] "
            f"loss={loss:.4f} | acc={metrics['accuracy']:.4f} | "
            f"precision={metrics['precision']:.4f} | recall={metrics['recall']:.4f} | "
            f"f1={metrics['f1_score']:.4f}"
        )
        history.append({
            "mode": label,
            "round": server_round,
            "loss": loss,
            **metrics,
        })
        return loss, metrics
 
    return evaluate

In [ ]:


# ## 7. Fungsi untuk Menjalankan Satu Eksperimen (IID atau Non-IID)
def run_experiment(mode: str) -> pd.DataFrame:
    """
    Menjalankan simulasi FedAvg penuh untuk satu mode partisi ("iid" atau "non_iid"),
    lalu mengembalikan DataFrame berisi metrik evaluasi global per round.
    """
    assert mode in ("iid", "non_iid")
    label = "IID" if mode == "iid" else f"Non-IID (alpha={DIRICHLET_ALPHA})"
 
    if mode == "iid":
        partitions = partition_iid(X_train_full, y_train_full, NUM_CLIENTS, seed=SEED)
    else:
        partitions = partition_non_iid_dirichlet(
            X_train_full, y_train_full, NUM_CLIENTS, alpha=DIRICHLET_ALPHA, seed=SEED
        )
 
    show_partition_stats(partitions, name=f"PARTISI - {label}")
 
    history: List[Dict[str, float]] = []
 
    client_app = ClientApp(client_fn=make_client_fn(partitions))
 
    def server_fn(context: Context) -> ServerAppComponents:
        initial_model = DrebinMLP(N_FEATURES, N_CLASSES)
        initial_parameters = ndarrays_to_parameters(get_parameters(initial_model))
 
        strategy = FedAvg(
            fraction_fit=1.0,
            fraction_evaluate=1.0,
            min_fit_clients=NUM_CLIENTS,
            min_evaluate_clients=NUM_CLIENTS,
            min_available_clients=NUM_CLIENTS,
            initial_parameters=initial_parameters,
            evaluate_fn=make_evaluate_fn(X_test, y_test, history, label),
            evaluate_metrics_aggregation_fn=weighted_average,
            fit_metrics_aggregation_fn=weighted_average,
        )
        config = ServerConfig(num_rounds=NUM_ROUNDS)
        return ServerAppComponents(strategy=strategy, config=config)
 
    server_app = ServerApp(server_fn=server_fn)
 
    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}
 
    print(f"\n{'#' * 70}\n# MENJALANKAN SIMULASI: {label}\n{'#' * 70}")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config=backend_config,
    )
 
    return pd.DataFrame(history)
 
 
# %% [markdown]
# ## 8. Jalankan Kedua Eksperimen & Bandingkan
 
# %%
if __name__ == "__main__":
    df_iid = run_experiment("iid")
    df_non_iid = run_experiment("non_iid")
 
    df_all = pd.concat([df_iid, df_non_iid], ignore_index=True)
 
    # =========================================================
    # Tabel hasil per round (semua metrik, kedua mode)
    # =========================================================
    print("\n" + "=" * 90)
    print("HASIL PER ROUND - IID vs NON-IID")
    print("=" * 90)
    print(
        df_all[["mode", "round", "loss", "accuracy", "precision", "recall", "f1_score"]]
        .to_string(index=False)
    )
 
    # =========================================================
    # Ringkasan: metrik pada round terakhir (kondisi akhir training)
    # =========================================================
    final_round = df_all["round"].max()
    df_final = df_all[df_all["round"] == final_round].copy()
 
    print("\n" + "=" * 90)
    print(f"RINGKASAN KINERJA AKHIR (Round {final_round})")
    print("=" * 90)
    print(
        df_final[["mode", "loss", "accuracy", "precision", "recall", "f1_score"]]
        .to_string(index=False)
    )
 
    # =========================================================
    # Ringkasan: rata-rata metrik selama seluruh training (semua round)
    # =========================================================
    df_avg = df_all.groupby("mode")[["loss", "accuracy", "precision", "recall", "f1_score"]].mean()
 
    print("\n" + "=" * 90)
    print(f"RATA-RATA KINERJA SELAMA {NUM_ROUNDS} ROUND")
    print("=" * 90)
    print(df_avg.to_string())
 
    # =========================================================
    # Simpan hasil ke CSV untuk dianalisis / diplot lebih lanjut
    # =========================================================
    output_path = "hasil_perbandingan_iid_vs_noniid.csv"
    df_all.to_csv(output_path, index=False)
    print(f"\n[INFO] Hasil lengkap per round disimpan ke: {output_path}")
 
    # =========================================================
    # (Opsional) Grafik perbandingan akurasi & f1-score per round
    # =========================================================
    try:
        import matplotlib.pyplot as plt
 
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
        for label, df_mode in df_all.groupby("mode"):
            axes[0].plot(df_mode["round"], df_mode["accuracy"], marker="o", label=label)
            axes[1].plot(df_mode["round"], df_mode["f1_score"], marker="o", label=label)
 
        axes[0].set_title("Akurasi Global per Round")
        axes[0].set_xlabel("Round")
        axes[0].set_ylabel("Accuracy")
        axes[0].legend()
        axes[0].grid(alpha=0.3)
 
        axes[1].set_title("F1-Score Global per Round")
        axes[1].set_xlabel("Round")
        axes[1].set_ylabel("F1-Score")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
 
        plt.tight_layout()
        fig_path = "perbandingan_iid_vs_noniid.png"
        plt.savefig(fig_path, dpi=150)
        print(f"[INFO] Grafik perbandingan disimpan ke: {fig_path}")
    except ImportError:
        print("[INFO] matplotlib tidak tersedia, grafik dilewati (tabel/CSV tetap tersimpan).")
 